In [58]:
# 1. Imports and data paths

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization import ImageNormalize, ZScaleInterval

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

MAPS_PATH = DATA_DIR / "manga-8138-12704-MAPS-HYB10-MILESHC-MASTARSSP.fits.gz"
DRP_CUBE_PATH = DATA_DIR / "manga-8138-12704-LOGCUBE.fits.gz"
DAP_CUBE_PATH = DATA_DIR / "manga-8138-12704-LOGCUBE-HYB10-MILESHC-MASTARSSP.fits.gz"

print("Project root:", PROJECT_ROOT)
for label, path in {
    "MAPS": MAPS_PATH,
    "DRP LOGCUBE": DRP_CUBE_PATH,
    "DAP LOGCUBE": DAP_CUBE_PATH,
}.items():
    exists = path.is_file()
    size_mb = path.stat().st_size / (1024**2) if exists else None
    print(f"{label:12s} | exists={exists} | size={size_mb:.2f} MB" if exists
          else f"{label:12s} | FILE NOT FOUND: {path}")

if not all(p.is_file() for p in [MAPS_PATH, DRP_CUBE_PATH, DAP_CUBE_PATH]):
    raise FileNotFoundError(
        "One or more MaNGA FITS files were not found. Check PROJECT_ROOT and DATA_DIR."
    )


Project root: d:\KS\manga-stellar-kinematics
MAPS         | exists=True | size=10.49 MB
DRP LOGCUBE  | exists=True | size=192.21 MB
DAP LOGCUBE  | exists=True | size=214.33 MB


In [59]:
# 2. Open the FITS products
# Keep these HDULists open while exploring their headers and data.
# If this cell is re-run, close any previously opened handles first.

for variable_name in ("maps", "drp_cube", "dap_cube"):
    old_hdul = globals().get(variable_name)
    if old_hdul is not None:
        try:
            old_hdul.close()
        except Exception:
            pass

maps = fits.open(MAPS_PATH, mode="readonly", memmap=False)
drp_cube = fits.open(DRP_CUBE_PATH, mode="readonly", memmap=False)
dap_cube = fits.open(DAP_CUBE_PATH, mode="readonly", memmap=False)

print("FITS products opened.")
print("MAPS HDU count:", len(maps))
print("DRP LOGCUBE HDU count:", len(drp_cube))
print("DAP LOGCUBE HDU count:", len(dap_cube))

print("\\nMAPS HDU inventory:")
maps.info()


FITS products opened.
MAPS HDU count: 64
DRP LOGCUBE HDU count: 24
DAP LOGCUBE HDU count: 13
\nMAPS HDU inventory:
Filename: d:\KS\manga-stellar-kinematics\data\raw\manga-8138-12704-MAPS-HYB10-MILESHC-MASTARSSP.fits.gz
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     147   ()      
  1  SPX_SKYCOO    1 ImageHDU        44   (72, 72, 2)   float32   
  2  SPX_ELLCOO    1 ImageHDU        49   (72, 72, 4)   float32   
  3  SPX_MFLUX     1 ImageHDU        37   (72, 72)   float32   
  4  SPX_MFLUX_IVAR    1 ImageHDU        38   (72, 72)   float32   
  5  SPX_SNR       1 ImageHDU        35   (72, 72)   float32   
  6  BINID         1 ImageHDU        46   (72, 72, 5)   int32   
  7  BIN_LWSKYCOO    1 ImageHDU        44   (72, 72, 2)   float32   
  8  BIN_LWELLCOO    1 ImageHDU        49   (72, 72, 4)   float32   
  9  BIN_AREA      1 ImageHDU        36   (72, 72)   float32   
 10  BIN_FAREA     1 ImageHDU        35   (72, 72)   float32   
 11  BI

In [60]:
# 3. Inspect primary headers

def show_primary_header(hdul, name):
    print(f"\\n{name} PRIMARY HEADER")
    print("-" * 72)
    header = hdul[0].header
    for key in [
        "TELESCOP", "INSTRUME", "PLATEID", "PLATEIFU", "MANGAID",
        "DAPTYPE", "VERSDRP2", "VERSDRP3", "VERSDAP", "DAPQUAL",
        "NAXIS", "EXTEND",
    ]:
        print(f"{key:12s}: {header.get(key, 'Not available')}")

show_primary_header(maps, "MAPS")
show_primary_header(drp_cube, "DRP LOGCUBE")
show_primary_header(dap_cube, "DAP LOGCUBE")


\nMAPS PRIMARY HEADER
------------------------------------------------------------------------
TELESCOP    : SDSS 2.5-M
INSTRUME    : MaNGA
PLATEID     : 8138
PLATEIFU    : 8138-12704
MANGAID     : 1-339041
DAPTYPE     : HYB10-MILESHC-MASTARSSP
VERSDRP2    : v3_1_1
VERSDRP3    : v3_1_1
VERSDAP     : 3.1.0
DAPQUAL     : 0
NAXIS       : 0
EXTEND      : True
\nDRP LOGCUBE PRIMARY HEADER
------------------------------------------------------------------------
TELESCOP    : SDSS 2.5-M
INSTRUME    : MaNGA
PLATEID     : 8138
PLATEIFU    : 8138-12704
MANGAID     : 1-339041
DAPTYPE     : Not available
VERSDRP2    : v3_1_1
VERSDRP3    : v3_1_1
VERSDAP     : Not available
DAPQUAL     : Not available
NAXIS       : 0
EXTEND      : True
\nDAP LOGCUBE PRIMARY HEADER
------------------------------------------------------------------------
TELESCOP    : SDSS 2.5-M
INSTRUME    : MaNGA
PLATEID     : 8138
PLATEIFU    : 8138-12704
MANGAID     : 1-339041
DAPTYPE     : HYB10-MILESHC-MASTARSSP
VERSDRP2    : v

In [61]:
# 4. Compare MaNGA metadata across products

def inspect_metadata(hdul, name):
    header = hdul[0].header
    keys = [
        "PLATEID", "PLATEIFU", "MANGAID", "IFUDSGN", "DAPTYPE",
        "VERSDRP2", "VERSDRP3", "VERSDAP",
        "OBJRA", "OBJDEC", "IFURA", "IFUDEC",
        "MNGTARG1", "MNGTARG2", "DAPQUAL",
    ]

    print(f"\\n{name}")
    print("-" * 68)
    for key in keys:
        print(f"{key:12s}: {header.get(key, 'Not available')}")

inspect_metadata(maps, "MAPS")
inspect_metadata(drp_cube, "DRP LOGCUBE")
inspect_metadata(dap_cube, "DAP LOGCUBE")


\nMAPS
--------------------------------------------------------------------
PLATEID     : 8138
PLATEIFU    : 8138-12704
MANGAID     : 1-339041
IFUDSGN     : 12704
DAPTYPE     : HYB10-MILESHC-MASTARSSP
VERSDRP2    : v3_1_1
VERSDRP3    : v3_1_1
VERSDAP     : 3.1.0
OBJRA       : 116.657135061
OBJDEC      : 44.7905163357
IFURA       : 116.65714
IFUDEC      : 44.790516
MNGTARG1    : 4160
MNGTARG2    : 0
DAPQUAL     : 0
\nDRP LOGCUBE
--------------------------------------------------------------------
PLATEID     : 8138
PLATEIFU    : 8138-12704
MANGAID     : 1-339041
IFUDSGN     : 12704
DAPTYPE     : Not available
VERSDRP2    : v3_1_1
VERSDRP3    : v3_1_1
VERSDAP     : Not available
OBJRA       : 116.657135061
OBJDEC      : 44.7905163357
IFURA       : 116.65714
IFUDEC      : 44.790516
MNGTARG1    : 4160
MNGTARG2    : 0
DAPQUAL     : Not available
\nDAP LOGCUBE
--------------------------------------------------------------------
PLATEID     : 8138
PLATEIFU    : 8138-12704
MANGAID     : 1-3390

In [62]:
# 5. Inspect stellar-kinematics extensions in the MAPS product

stellar_extensions = [
    "STELLAR_VEL",
    "STELLAR_SIGMA",
    "STELLAR_SIGMACORR",
    "STELLAR_VEL_IVAR",
    "STELLAR_SIGMA_IVAR",
    "STELLAR_VEL_MASK",
    "STELLAR_SIGMA_MASK",
]

available_maps = {hdu.name.upper(): hdu for hdu in maps}

print("Available MAPS extensions:")
for index, hdu in enumerate(maps):
    shape = hdu.data.shape if hdu.data is not None else "No data"
    print(f"{index:02d} | {hdu.name:25s} | {shape}")

print("\\nRequested stellar extensions:")
for name in stellar_extensions:
    hdu = available_maps.get(name)
    if hdu is None:
        print(f"\\n{name}: NOT FOUND in this MAPS file")
        continue

    shape = hdu.data.shape if hdu.data is not None else "No data"
    print(f"\\nExtension: {name}")
    print("Shape:", shape)
    print("BUNIT:", hdu.header.get("BUNIT", "Not specified"))
    print("MASKNAME:", hdu.header.get("MASKNAME", "Not specified"))
    print("Description:", hdu.header.get("HDUCLAS2", "Not specified"))

# Important diagnostic:
# A MAPS file with only PRIMARY is not a usable multi-extension MAPS product.
if len(maps) == 1:
    print(
        "\\nWARNING: This opened MAPS file contains only PRIMARY. "
        "The stellar maps cannot be extracted from this file. "
        "Check the file path, file size, and whether the download is complete."
    )


Available MAPS extensions:
00 | PRIMARY                   | No data
01 | SPX_SKYCOO                | (2, 72, 72)
02 | SPX_ELLCOO                | (4, 72, 72)
03 | SPX_MFLUX                 | (72, 72)
04 | SPX_MFLUX_IVAR            | (72, 72)
05 | SPX_SNR                   | (72, 72)
06 | BINID                     | (5, 72, 72)
07 | BIN_LWSKYCOO              | (2, 72, 72)
08 | BIN_LWELLCOO              | (4, 72, 72)
09 | BIN_AREA                  | (72, 72)
10 | BIN_FAREA                 | (72, 72)
11 | BIN_MFLUX                 | (72, 72)
12 | BIN_MFLUX_IVAR            | (72, 72)
13 | BIN_MFLUX_MASK            | (72, 72)
14 | BIN_SNR                   | (72, 72)
15 | STELLAR_VEL               | (72, 72)
16 | STELLAR_VEL_IVAR          | (72, 72)
17 | STELLAR_VEL_MASK          | (72, 72)
18 | STELLAR_SIGMA             | (72, 72)
19 | STELLAR_SIGMA_IVAR        | (72, 72)
20 | STELLAR_SIGMA_MASK        | (72, 72)
21 | STELLAR_SIGMACORR         | (2, 72, 72)
22 | STELLAR_FOM               |

In [63]:
def inspect_extensions(hdul, product_name, candidates=None):
    print(f"\\n{product_name} extensions")
    print("=" * 76)
    for index, hdu in enumerate(hdul):
        shape = hdu.data.shape if hdu.data is not None else "No data"
        unit = hdu.header.get("BUNIT", "Not specified")
        print(f"{index:02d} | {hdu.name:20s} | shape={shape} | unit={unit}")

    if candidates:
        names = {hdu.name.upper() for hdu in hdul}
        print("\\nCandidate extension availability:")
        for name in candidates:
            print(f"{name:15s}: {'available' if name.upper() in names else 'not present'}")

inspect_extensions(
    drp_cube,
    "DRP LOGCUBE",
    candidates=["FLUX", "IVAR", "MASK", "WAVE", "LSF", "LSFPRE", "LSFPOST"],
)
inspect_extensions(
    dap_cube,
    "DAP LOGCUBE",
    candidates=["FLUX", "IVAR", "MASK", "WAVE", "MODEL", "EMLINE"],
)
                                                

\nDRP LOGCUBE extensions
00 | PRIMARY              | shape=No data | unit=1E-17 erg/s/cm^2/Angstrom/spaxel
01 | FLUX                 | shape=(4563, 72, 72) | unit=1E-17 erg/s/cm^2/Angstrom/spaxel
02 | IVAR                 | shape=(4563, 72, 72) | unit=Not specified
03 | MASK                 | shape=(4563, 72, 72) | unit=Not specified
04 | LSFPOST              | shape=(4563, 72, 72) | unit=Not specified
05 | LSFPRE               | shape=(4563, 72, 72) | unit=Not specified
06 | WAVE                 | shape=(4563,) | unit=Not specified
07 | SPECRES              | shape=(4563,) | unit=Not specified
08 | SPECRESD             | shape=(4563,) | unit=Not specified
09 | PRESPECRES           | shape=(4563,) | unit=Not specified
10 | PRESPECRESD          | shape=(4563,) | unit=Not specified
11 | OBSINFO              | shape=(15,) | unit=Not specified
12 | GIMG                 | shape=(72, 72) | unit=nanomaggies/pixel
13 | RIMG                 | shape=(72, 72) | unit=nanomaggies/pixel
14 | IIMG   

In [64]:
for hdul in (maps, drp_cube, dap_cube):
    hdul.close()

print("All FITS files closed.")


All FITS files closed.
